In [1]:
from utils import load_and_prepare_data, quantize_fp16_dynamic, benchmark_cpu, quantize_hidden_only, quantize_dynamic_model, quantize_all_but_output
from neural_network import NeuralNetwork
from sklearn.metrics import classification_report
import torch

numerical_cols = [
        "duration",
        "dst_bytes",
        "missed_bytes",
        "src_bytes",
        "src_ip_bytes",
        "src_pkts",
        "dst_pkts",
        "dst_ip_bytes",
        "http_request_body_len",
        "http_response_body_len"

    ]

categorical_cols = [
        "proto",
        "conn_state",
        "http_status_code",
        "http_method",
        "http_orig_mime_types",
        "http_resp_mime_types",
    ]


target_col = 'type'
num_target_classes = 8
dataset_path = 'datasets/http_ton.csv'
batch_size = 2048
epochs = 10
values_to_remove = {'type': ['mitm', 'dos']}

In [2]:
train_dataloader, valid_dataloader, test_dataloader, cat_cardinalities, cw, target_names = load_and_prepare_data(
    file_path=dataset_path,
    target_col=target_col,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    batch_size=batch_size,
    rows_to_remove=values_to_remove
)

embedding_dims = [min(50, (card + 1) // 2) for card in cat_cardinalities]

In [3]:
# model = NeuralNetwork(
#     hidden_layers_sizes=[256, 256, 256], 
#     cat_cardinalities=cat_cardinalities,
#     embedding_dims=embedding_dims,
#     num_numerical_features=len(numerical_cols),
#     num_target_classes=num_target_classes,
# )

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# model.fit(
#     train_dataloader=train_dataloader,
#     valid_dataloader=valid_dataloader,
#     device=device,
#     optimizer=optimizer,
#     lr_scheduler=scheduler,
#     epochs=epochs,
# )

In [6]:
model = NeuralNetwork.load('best_model.pt', device=device)

[WARNING] CUDA non disponibile. Caricamento su CPU.


In [7]:
model_quant = quantize_all_but_output(model)

Platform: x86_64


In [8]:
std_times = benchmark_cpu(model, test_dataloader, num_threads=1)
print(f"Standard Model:\n {std_times}")
print("--------------------")
quant_times = benchmark_cpu(model_quant, test_dataloader, num_threads=1)
print(f"Quant model:\n {quant_times}")

Standard Model:
 {'batch_size': 2048, 'num_threads': 1, 'median_ms': 7.0332834998225735, 'p95_ms': 9.336410600326415, 'throughput_sps': 281377.7532528828}
--------------------
Quant model:
 {'batch_size': 2048, 'num_threads': 1, 'median_ms': 5.049217499617953, 'p95_ms': 9.590817500156836, 'throughput_sps': 375964.10032089095}


In [9]:
std = std_times['throughput_sps']
qnt = quant_times['throughput_sps']

latency_reduction_pct = (1 - std / qnt) * 100
print(f"Riduzione tempo di inferenza: {latency_reduction_pct:.1f}%")

Riduzione tempo di inferenza: 25.2%


In [10]:
model_preds = model.predict(test_dataloader,device)
quant_preds = model_quant.predict(test_dataloader, device)

In [11]:
y_true = torch.cat([y for _, _, y in test_dataloader]).numpy()
print("\n=== Classification Report DNN===")
print(classification_report(y_true, model_preds.numpy(), target_names=target_names, digits=4))
print("\n=== Classification Report Quant===")
print(classification_report(y_true, quant_preds.numpy(), target_names=target_names, digits=4))



=== Classification Report DNN===
              precision    recall  f1-score   support

        ddos     0.9572    0.9559    0.9566     50615
   injection     0.9594    0.9146    0.9364     50967
      normal     0.9913    0.7411    0.8481      9197
    password     0.9766    0.9934    0.9849    189474
    scanning     0.9944    0.9174    0.9544      4686
         xss     0.9750    0.9836    0.9793    211140

    accuracy                         0.9727    516079
   macro avg     0.9756    0.9177    0.9433    516079
weighted avg     0.9727    0.9727    0.9723    516079


=== Classification Report Quant===
              precision    recall  f1-score   support

        ddos     0.1523    0.4254    0.2243     50615
   injection     0.2551    0.0748    0.1157     50967
      normal     0.0248    0.3035    0.0459      9197
    password     0.5825    0.2778    0.3762    189474
    scanning     0.8631    0.8596    0.8613      4686
         xss     0.7861    0.5670    0.6588    211140

    acc